# Questao 5 - Análise de clientes

### 1. Configuração e Limpeza de Categorias (Python)

In [1]:
# Importando bibliotecas
import pandas as pd
import sqlite3
import json

In [2]:
# Carregando os dados
df_vendas = pd.read_csv('datasets/vendas_2023_2024.csv')
df_produtos = pd.read_csv('datasets/produtos_raw.csv')

In [3]:
# --- Parte 1: Limpeza das Categorias (Lógica de radicais) ---
def limpar_categoria(txt):
    txt = str(txt).lower().replace(" ", "")
    if 'eletr' in txt: return 'eletrônicos'
    if 'prop' in txt: return 'propulsão'
    if 'ancor' in txt or 'encor' in txt: return 'ancoragem'
    return 'outros'

df_produtos['category_clean'] = df_produtos['actual_category'].apply(limpar_categoria)

In [4]:
df_produtos.head(5)

,name,price,code,actual_category,category_clean
0,Transponder AIS Maré Magnum,R$ 33122.52,1,ELETRONICOS,eletrônicos
1,Transponder Furuno Marlin,R$ 13998.15,2,ELETRONICOS,eletrônicos
2,Radar Furuno Pulse Leviathan,R$ 9024.19,3,E L E T R Ô N I C O S,eletrônicos
3,Rádio AIS Hydro Tidal Zen,R$ 3381.88,4,Eletrunicos,eletrônicos
4,Piloto Automático Furuno Storm,R$ 23669.01,5,Eletronicoz,eletrônicos


In [5]:
# Criando conexão SQL em memória para a análise
conn = sqlite3.connect(':memory:')

### 2. Carregamento e Integração dos Clientes (Python)

In [6]:
# 1. Carregar o arquivo JSON de clientes
with open('datasets/clientes_crm.json', 'r', encoding='utf-8') as f:
    dados_clientes = json.load(f)

# 2. Transformar em DataFrame
df_clientes_master = pd.DataFrame(dados_clientes)

df_clientes_master.head(5)

,full_name,location,code,email
0,Femininos Oliveira Antunes,"Aratu (Candeias) , BA",1,femininos.oliveira.antunes@icloud.com
1,Fernanda Azevedo Soares Nunes Vieira,"PE , Recife",2,nunes.fernanda.soares.azevedo.vieira@outlook.com
2,Daniel Farias Ribeiro Teixeira,"Rio Grande,RS",3,farias.teixeira.daniel.ribeiro#gmail.com
3,Thiago Moreira,"AC , Rio Branco",4,thiago.moreira#gmail.com
4,Pedro Freitas,PA - Santarém Novo,5,pedro.freitas#icloud.com


In [7]:
# 3. Criar a conexão ÚNICA e carregar as 3 tabelas
conn = sqlite3.connect(':memory:')

df_vendas.to_sql('vendas', conn, index=False, if_exists='replace')
df_produtos.to_sql('produtos', conn, index=False, if_exists='replace')
df_clientes_master.to_sql('clientes_master', conn, index=False, if_exists='replace')

print("Sucesso: Tabelas 'vendas', 'produtos' e 'clientes_master' prontas para consulta.")

Sucesso: Tabelas 'vendas', 'produtos' e 'clientes_master' prontas para consulta.


### 3. Questão 5.1 — Código SQL Consolidado

In [8]:
query_relatorio_elite = """
WITH Metricas_Clientes AS (
    SELECT 
        v.id_client,
        SUM(v.total) AS faturamento_total,
        COUNT(v.id) AS frequencia,
        (SUM(v.total) / COUNT(v.id)) AS ticket_medio,
        COUNT(DISTINCT p.category_clean) AS diversidade_categorias
    FROM vendas v
    JOIN produtos p ON v.id_product = p.code
    GROUP BY v.id_client
),
Top_10_Fieis AS (
    -- Aplica o filtro de elite: 3+ categorias e Top 10 Ticket Médio
    SELECT id_client, ticket_medio, diversidade_categorias
    FROM Metricas_Clientes
    WHERE diversidade_categorias >= 3
    ORDER BY ticket_medio DESC, id_client ASC
    LIMIT 10
)
-- Une com os dados cadastrais do JSON para o relatório final
SELECT 
    c.full_name,
    c.location,
    t.ticket_medio,
    t.diversidade_categorias
FROM Top_10_Fieis t
JOIN clientes_master c ON t.id_client = c.code;
"""




In [9]:
df_elite_final = pd.read_sql(query_relatorio_elite, conn)
df_elite_final

,full_name,location,ticket_medio,diversidade_categorias
0,Gabriela Barros Lacerda,RJ / Niterói,334991.747772,3
1,Márcia Figueiredo,Vila do Conde (Barcarena) - PA,319059.596336,3
2,Ana Silva Costa Farias Coelho,Macapá/AP,311805.458867,3
3,Lucas Guedes Cunha Lopes,PB - João Pessoa,310672.413889,3
4,Francisca Ribeiro Pinheiro,Belém/PA,301521.448696,3
5,Fernanda Azevedo Soares Nunes Vieira,"PE , Recife",300964.690000,3
6,Daniela Borges Vieira Farias Mendonça,SE / Aracaju,298318.472000,3
7,Bianca Rodrigues,"PR , Antonina",298268.631136,3
8,Luiz Borges Gomes Araújo,Manaus/AM,297558.483036,3
9,Pedro Freitas,PA - Santarém Novo,290815.536667,3


### 4. Validação da Categoria Mais Vendida (Questão 5.2)

In [10]:
# Consulta para descobrir a categoria preferida da elite
query_validacao_52 = """
WITH Top_10_Elite_IDs AS (
    -- Reutilizamos a lógica para pegar apenas os IDs dos 10 clientes fieis
    SELECT v.id_client
    FROM vendas v
    JOIN produtos p ON v.id_product = p.code
    GROUP BY v.id_client
    HAVING COUNT(DISTINCT p.category_clean) >= 3
    ORDER BY (SUM(v.total) / COUNT(v.id)) DESC, v.id_client ASC
    LIMIT 10
)
SELECT 
    p.category_clean,
    SUM(v.qtd) AS quantidade_total_itens
FROM vendas v
JOIN produtos p ON v.id_product = p.code
WHERE v.id_client IN (SELECT id_client FROM Top_10_Elite_IDs)
GROUP BY p.category_clean
ORDER BY quantidade_total_itens DESC;
"""


In [11]:
df_validacao_categoria = pd.read_sql(query_validacao_52, conn)
df_validacao_categoria

,category_clean,quantidade_total_itens
0,propulsão,6429
1,ancoragem,6003
2,eletrônicos,5376


### Relatório final (Questão 5.3)

- ***Como você realizou a limpeza das categorias?*** Durante uma análise exploratória, notei um padrão de erros na grafia das categorias, e visando padronizá-las, eu utilizei uma função Python baseada em radicais de texto (ex: eletr, prop, ancor). Isso permitiu agrupar variações como "Eletronicoz", "E L E T R Ô N I C O S" e "Eletrunicos" sob o rótulo único de eletrônicos, garantindo que a diversidade de compra fosse calculada sem erros de duplicidade por erro de grafia.

- ***Qual lógica utilizou para filtrar os clientes com diversidade mínima?*** Utilizei o agrupamento por cliente combinando as funções COUNT(DISTINCT category_clean) e a cláusula HAVING >= 3. Essa lógica isolou apenas os clientes que transacionaram em todos os principais departamentos da loja, atendendo ao critério de "navegação por diversas categorias" exigido pela diretoria.

- ***Como garantiu que a contagem de itens refletisse apenas os Top 10?*** Utilizei uma subquery para primeiro fixar o ranking dos 10 maiores Ticket Médios. No comando final, apliquei um filtro WHERE id_client IN (...) que restringiu o cálculo da SUM(qtd) exclusivamente para o comportamento de compra deste grupo seleto, impedindo que o volume de clientes comuns distorcesse o resultado.